In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

In [4]:
# 현재 노트북이 notebooks 폴더 안에서 실행된다는 기준
NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent

# 깃허브 프로젝트 폴더 옆의 project_data
DATA_DIR = PROJECT_DIR.parent / "project_data"

RAW_DIR = DATA_DIR / "raw"
ALL_AGE_DIR = RAW_DIR / "전연령"
PROCESSED_DIR = DATA_DIR / "processed"
REFERENCE_DIR = DATA_DIR / "reference"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("현재 위치:", NOTEBOOK_DIR)
print("프로젝트 폴더:", PROJECT_DIR)
print("전체연령 데이터 폴더:", ALL_AGE_DIR)
print("결과 저장 폴더:", PROCESSED_DIR)

print("\n폴더 존재 여부")
print("ALL_AGE_DIR:", ALL_AGE_DIR.exists())
print("PROCESSED_DIR:", PROCESSED_DIR.exists())

현재 위치: /Users/janghwayeong/Desktop/부트캠프_프로젝트/Analysis-of-Youth-Savings-Capacity-Project/notebooks
프로젝트 폴더: /Users/janghwayeong/Desktop/부트캠프_프로젝트/Analysis-of-Youth-Savings-Capacity-Project
전체연령 데이터 폴더: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/raw/전연령
결과 저장 폴더: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed

폴더 존재 여부
ALL_AGE_DIR: True
PROCESSED_DIR: True


In [6]:
from pathlib import Path

print("현재 실행 위치:", Path.cwd())
print("현재 폴더 내용:")

for path in Path.cwd().iterdir():
    print(path.name)

현재 실행 위치: /Users/janghwayeong/Desktop/부트캠프_프로젝트/Analysis-of-Youth-Savings-Capacity-Project/notebooks
현재 폴더 내용:
02_calculate_youth_single_household_share.ipynb
03_inspect_youth_mobility.ipynb
01_preprocess_youth_single_household.ipynb
04_build_commute_od.ipynb


In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
import re


# =========================================================
# 1. project_data 폴더 자동 탐색
# =========================================================

CURRENT_DIR = Path.cwd().resolve()

candidate_dirs = [
    CURRENT_DIR / "project_data",
    CURRENT_DIR.parent / "project_data",
    CURRENT_DIR.parent.parent / "project_data",
]

DATA_DIR = next(
    (path for path in candidate_dirs if path.exists()),
    None
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "project_data 폴더를 찾지 못했습니다.\n"
        f"현재 실행 위치: {CURRENT_DIR}"
    )

RAW_DIR = DATA_DIR / "raw"
ALL_AGE_DIR = RAW_DIR / "전연령"
PROCESSED_DIR = DATA_DIR / "processed"
REFERENCE_DIR = DATA_DIR / "reference"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("현재 실행 위치:", CURRENT_DIR)
print("DATA_DIR:", DATA_DIR)
print("RAW_DIR:", RAW_DIR)
print("ALL_AGE_DIR:", ALL_AGE_DIR)

print("\n폴더 존재 여부")
print("DATA_DIR:", DATA_DIR.exists())
print("RAW_DIR:", RAW_DIR.exists())
print("ALL_AGE_DIR:", ALL_AGE_DIR.exists())
print("PROCESSED_DIR:", PROCESSED_DIR.exists())

현재 실행 위치: /Users/janghwayeong/Desktop/부트캠프_프로젝트/Analysis-of-Youth-Savings-Capacity-Project/notebooks
DATA_DIR: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data
RAW_DIR: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/raw
ALL_AGE_DIR: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/raw/전연령

폴더 존재 여부
DATA_DIR: True
RAW_DIR: True
ALL_AGE_DIR: True
PROCESSED_DIR: True


In [12]:
print("전연령 폴더 안의 전체 항목")

for file in ALL_AGE_DIR.iterdir():
    print(
        "파일명:",
        repr(file.name),
        "| 파일 여부:",
        file.is_file(),
        "| 확장자:",
        repr(file.suffix)
    )

전연령 폴더 안의 전체 항목
파일명: '1월 출근 목적 이동 데이터.csv' | 파일 여부: True | 확장자: '.csv'
파일명: '3월 출근 목적 이동 데이터.csv' | 파일 여부: True | 확장자: '.csv'
파일명: '2월 출근 목적 이동 데이터.csv' | 파일 여부: True | 확장자: '.csv'
파일명: '4월 출근 목적 이동 데이터.csv' | 파일 여부: True | 확장자: '.csv'
파일명: '12월 출근 목적 이동 데이터.csv' | 파일 여부: True | 확장자: '.csv'
파일명: '9월 출근 목적 이동 데이터.csv' | 파일 여부: True | 확장자: '.csv'
파일명: '10월 출근 목적 이동 데이터.csv' | 파일 여부: True | 확장자: '.csv'
파일명: '6월 출근 목적 이동 데이터.csv' | 파일 여부: True | 확장자: '.csv'
파일명: '5월 출근 목적 이동 데이터.csv' | 파일 여부: True | 확장자: '.csv'
파일명: '8월 출근 목적 이동 데이터.csv' | 파일 여부: True | 확장자: '.csv'
파일명: '11월 출근 목적 이동 데이터.csv' | 파일 여부: True | 확장자: '.csv'
파일명: '7월 출근 목적 이동 데이터.csv' | 파일 여부: True | 확장자: '.csv'


In [15]:
import re
import unicodedata

In [16]:
def normalize_name(text):
    return unicodedata.normalize("NFC", text)


def get_month(file_path):
    file_name = normalize_name(file_path.name)
    match = re.search(r"(\d+)월", file_name)

    return int(match.group(1)) if match else 999


commute_files = sorted(
    [
        file
        for file in ALL_AGE_DIR.iterdir()
        if file.is_file()
        and file.suffix.lower() == ".csv"
        and "출근" in normalize_name(file.name)
    ],
    key=get_month
)

print("출근 목적 파일 수:", len(commute_files))

for file in commute_files:
    print(normalize_name(file.name))

출근 목적 파일 수: 12
1월 출근 목적 이동 데이터.csv
2월 출근 목적 이동 데이터.csv
3월 출근 목적 이동 데이터.csv
4월 출근 목적 이동 데이터.csv
5월 출근 목적 이동 데이터.csv
6월 출근 목적 이동 데이터.csv
7월 출근 목적 이동 데이터.csv
8월 출근 목적 이동 데이터.csv
9월 출근 목적 이동 데이터.csv
10월 출근 목적 이동 데이터.csv
11월 출근 목적 이동 데이터.csv
12월 출근 목적 이동 데이터.csv


In [17]:
sample_file = commute_files[0]

encodings = ["utf-8-sig", "utf-8", "cp949", "euc-kr"]

sample = None
used_encoding = None

for encoding in encodings:
    try:
        sample = pd.read_csv(
            sample_file,
            encoding=encoding,
            nrows=5
        )

        used_encoding = encoding
        break

    except UnicodeDecodeError:
        continue

if sample is None:
    raise ValueError("CSV 인코딩을 확인하지 못했습니다.")

print("확인 파일:", normalize_name(sample_file.name))
print("사용 인코딩:", used_encoding)
print("컬럼 수:", len(sample.columns))

for index, column in enumerate(sample.columns, start=1):
    print(f"{index}. {repr(column)}")

display(sample)

확인 파일: 1월 출근 목적 이동 데이터.csv
사용 인코딩: utf-8-sig
컬럼 수: 12
1. '출발 행정동 코드'
2. '출발 행정동'
3. '도착 행정동 코드'
4. '도착 행정동'
5. '출발 시간 코드'
6. '도착 시간 코드'
7. '이동 목적 코드'
8. '이동거리'
9. '이동 시간'
10. '이동 인구수'
11. '데이터 기준일'
12. '요일'


,출발 행정동 코드,출발 행정동,도착 행정동 코드,도착 행정동,출발 시간 코드,도착 시간 코드,이동 목적 코드,이동거리,이동 시간,이동 인구수,데이터 기준일,요일
0,11110515,청운효자동,11110615,종로1.2.3.4가동,00:00,00:00,1,955,0,3.62,2025-01-01,수
1,11110530,사직동,11110530,사직동,00:00,00:00,1,276,0,7.33,2025-01-01,수
2,11110530,사직동,11200535,왕십리도선동,00:00,00:00,1,4159,20,1.88,2025-01-01,수
3,11110560,평창동,11290525,성북동,00:00,00:00,1,894,4,2.99,2025-01-01,수
4,11110570,무악동,11170510,후암동,00:00,00:00,1,3070,11,4.09,2025-01-01,수


In [19]:
import pandas as pd
import numpy as np
from pathlib import Path
import unicodedata


# 현재 확인한 sample_file과 같은 폴더에
# 1월~12월 CSV가 있다고 가정
ALL_AGE_DIR = sample_file.parent


def normalize_name(name):
    return unicodedata.normalize("NFC", str(name)).strip()


def clean_columns(df):
    df = df.copy()

    df.columns = [
        normalize_name(column)
        .replace("\n", "")
        .replace("\r", "")
        .strip()
        for column in df.columns
    ]

    return df


def read_csv_auto(file_path):
    encodings = [
        "utf-8-sig",
        "cp949",
        "euc-kr",
        "utf-8",
    ]

    for encoding in encodings:
        try:
            df = pd.read_csv(
                file_path,
                encoding=encoding,
                dtype={
                    "출발 행정동 코드": "string",
                    "도착 행정동 코드": "string",
                },
                low_memory=False,
            )

            return clean_columns(df), encoding

        except UnicodeDecodeError:
            continue

    raise ValueError(
        f"인코딩을 확인하지 못했습니다: {file_path.name}"
    )


monthly_files = sorted(
    ALL_AGE_DIR.glob("*.csv"),
    key=lambda path: normalize_name(path.name),
)


print("CSV 파일 수:", len(monthly_files))

for file_path in monthly_files:
    print("-", normalize_name(file_path.name))

CSV 파일 수: 12
- 10월 출근 목적 이동 데이터.csv
- 11월 출근 목적 이동 데이터.csv
- 12월 출근 목적 이동 데이터.csv
- 1월 출근 목적 이동 데이터.csv
- 2월 출근 목적 이동 데이터.csv
- 3월 출근 목적 이동 데이터.csv
- 4월 출근 목적 이동 데이터.csv
- 5월 출근 목적 이동 데이터.csv
- 6월 출근 목적 이동 데이터.csv
- 7월 출근 목적 이동 데이터.csv
- 8월 출근 목적 이동 데이터.csv
- 9월 출근 목적 이동 데이터.csv


In [20]:
monthly_data = []
read_log = []


for file_path in monthly_files:
    df, encoding = read_csv_auto(file_path)

    df["원본 파일명"] = normalize_name(file_path.name)

    monthly_data.append(df)

    read_log.append({
        "파일명": normalize_name(file_path.name),
        "인코딩": encoding,
        "행 수": len(df),
        "열 수": len(df.columns),
    })


all_age_raw = pd.concat(
    monthly_data,
    ignore_index=True,
)


read_log_df = pd.DataFrame(read_log)


print("전체 결합 완료")
print("전체 행 수:", f"{len(all_age_raw):,}")
print("전체 열 수:", len(all_age_raw.columns))

display(read_log_df)
display(all_age_raw.head())

전체 결합 완료
전체 행 수: 139,027,086
전체 열 수: 13


,파일명,인코딩,행 수,열 수
0,10월 출근 목적 이동 데이터.csv,utf-8-sig,10968894,13
1,11월 출근 목적 이동 데이터.csv,utf-8-sig,11324717,13
2,12월 출근 목적 이동 데이터.csv,utf-8-sig,12554450,13
3,1월 출근 목적 이동 데이터.csv,utf-8-sig,10189569,13
4,2월 출근 목적 이동 데이터.csv,utf-8-sig,10757916,13
5,3월 출근 목적 이동 데이터.csv,utf-8-sig,11314600,13
6,4월 출근 목적 이동 데이터.csv,utf-8-sig,12327147,13
7,5월 출근 목적 이동 데이터.csv,utf-8-sig,11428982,13
8,6월 출근 목적 이동 데이터.csv,utf-8-sig,11237334,13
9,7월 출근 목적 이동 데이터.csv,utf-8-sig,13012623,13


,출발 행정동 코드,출발 행정동,도착 행정동 코드,도착 행정동,출발 시간 코드,도착 시간 코드,이동 목적 코드,이동거리,이동 시간,이동 인구수,데이터 기준일,요일,원본 파일명
0,11110530,사직동,11110530,사직동,00:00,00:00,1,424,4,2.48,2025-10-01,수,10월 출근 목적 이동 데이터.csv
1,11110530,사직동,11410520,천연동,00:00,00:00,1,1334,11,4.04,2025-10-01,수,10월 출근 목적 이동 데이터.csv
2,11110550,부암동,11110550,부암동,00:00,00:00,1,250,9,2.64,2025-10-01,수,10월 출근 목적 이동 데이터.csv
3,11110615,종로1.2.3.4가동,11110615,종로1.2.3.4가동,00:00,00:00,1,640,2,1.21,2025-10-01,수,10월 출근 목적 이동 데이터.csv
4,11110630,종로5.6가동,11110630,종로5.6가동,00:00,00:00,1,320,2,2.47,2025-10-01,수,10월 출근 목적 이동 데이터.csv


In [21]:
# 이미 출근 목적만 남아 있는 전체연령 데이터
commute_raw = all_age_raw.copy()

print("출근 데이터 행 수:", f"{len(commute_raw):,}")
display(commute_raw.head())

출근 데이터 행 수: 139,027,086


,출발 행정동 코드,출발 행정동,도착 행정동 코드,도착 행정동,출발 시간 코드,도착 시간 코드,이동 목적 코드,이동거리,이동 시간,이동 인구수,데이터 기준일,요일,원본 파일명
0,11110530,사직동,11110530,사직동,00:00,00:00,1,424,4,2.48,2025-10-01,수,10월 출근 목적 이동 데이터.csv
1,11110530,사직동,11410520,천연동,00:00,00:00,1,1334,11,4.04,2025-10-01,수,10월 출근 목적 이동 데이터.csv
2,11110550,부암동,11110550,부암동,00:00,00:00,1,250,9,2.64,2025-10-01,수,10월 출근 목적 이동 데이터.csv
3,11110615,종로1.2.3.4가동,11110615,종로1.2.3.4가동,00:00,00:00,1,640,2,1.21,2025-10-01,수,10월 출근 목적 이동 데이터.csv
4,11110630,종로5.6가동,11110630,종로5.6가동,00:00,00:00,1,320,2,2.47,2025-10-01,수,10월 출근 목적 이동 데이터.csv


In [ ]:
# =========================================================
# 출발 시간 코드 확인
# =========================================================

TIME_COL = "출발 시간 코드"

print("전체 행 수:", f"{len(commute_raw):,}")
print("시간 컬럼 dtype:", commute_raw[TIME_COL].dtype)
print("시간 결측치:", f"{commute_raw[TIME_COL].isna().sum():,}")

# 시간 코드 종류 확인
time_values = (
    commute_raw[TIME_COL]
    .dropna()
    .value_counts()
    .sort_index()
)

print("\n시간 코드 개수:", len(time_values))
display(time_values.head(30))

전체 행 수: 139,027,086
시간 컬럼 dtype: str
시간 결측치: 0

시간 코드 개수: 36


출발 시간 코드
00:00      103691
01:00      248477
02:00      221089
03:00      273024
04:00      690747
05:00     3632550
06:00    14858132
07:00    10623613
07:20    12633196
07:40    13613896
08:00    13288042
08:20    11116323
08:40     8765935
09:00     7333669
09:20     5561163
09:40     3968511
10:00     6748352
11:00     5331610
12:00     5217058
13:00     4327168
14:00     3258357
15:00     2504064
16:00     1662115
17:00      435917
17:20      364445
17:40      301700
18:00      256246
18:20      216038
18:40      194036
19:00      175304
Name: count, dtype: int64

In [23]:
import pandas as pd
import numpy as np

# =========================================================
# 1. 데이터 기본 구조 확인
# =========================================================

print("=" * 60)
print("데이터 기본 정보")
print("=" * 60)

print("행 수:", f"{len(commute_raw):,}")
print("열 수:", commute_raw.shape[1])

print("\n컬럼 목록")
for i, col in enumerate(commute_raw.columns, start=1):
    print(f"{i:>2}. {col} ({commute_raw[col].dtype})")


# =========================================================
# 결측치 확인
# =========================================================

missing_check = pd.DataFrame({
    "결측치 수": commute_raw.isna().sum(),
    "결측률(%)": (
        commute_raw.isna().mean() * 100
    ).round(4)
})

missing_check = (
    missing_check
    .sort_values("결측치 수", ascending=False)
)

print("\n결측치가 있는 컬럼")
display(
    missing_check[
        missing_check["결측치 수"] > 0
    ]
)

데이터 기본 정보
행 수: 139,027,086
열 수: 13

컬럼 목록
 1. 출발 행정동 코드 (string)
 2. 출발 행정동 (str)
 3. 도착 행정동 코드 (string)
 4. 도착 행정동 (str)
 5. 출발 시간 코드 (str)
 6. 도착 시간 코드 (str)
 7. 이동 목적 코드 (int64)
 8. 이동거리 (int64)
 9. 이동 시간 (int64)
10. 이동 인구수 (float64)
11. 데이터 기준일 (str)
12. 요일 (str)
13. 원본 파일명 (str)

결측치가 있는 컬럼


,결측치 수,결측률(%)


In [24]:
# =========================================================
# 2. 숫자형 변수 기초 통계
# =========================================================

numeric_cols = commute_raw.select_dtypes(
    include="number"
).columns.tolist()

print("숫자형 컬럼:", numeric_cols)

if numeric_cols:
    numeric_summary = (
        commute_raw[numeric_cols]
        .describe(
            percentiles=[
                0.01,
                0.05,
                0.25,
                0.50,
                0.75,
                0.95,
                0.99
            ]
        )
        .T
    )

    display(numeric_summary)
else:
    print("숫자형 컬럼이 없습니다.")

숫자형 컬럼: ['이동 목적 코드', '이동거리', '이동 시간', '이동 인구수']


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
이동 목적 코드,139027086.0,1.000000,0.000000,1.0,1.00,1.00,1.0,1.0,1.00,1.00,1.00,1.0
이동거리,139027086.0,6815.499703,5351.942073,0.0,187.00,450.00,2394.0,5661.0,10105.00,17175.00,22038.00,34384.0
이동 시간,139027086.0,45.244751,40.667322,0.0,0.00,3.00,21.0,40.0,59.00,102.00,186.00,1334.0
이동 인구수,139027086.0,4.833654,6.393368,1.0,1.83,2.24,2.9,3.5,4.29,11.35,25.39,1536.5


In [26]:
# 문자열 형태로 저장된 숫자형 후보 확인

for keyword in ["시간", "거리", "이동량", "인구"]:
    matched_cols = [
        col for col in commute_raw.columns
        if keyword in col
    ]

    print(f"\n[{keyword}] 관련 컬럼")
    for col in matched_cols:
        print(
            col,
            "| dtype:", commute_raw[col].dtype,
            "| 예시:",
            commute_raw[col].dropna().head(5).tolist()
        )


[시간] 관련 컬럼
출발 시간 코드 | dtype: str | 예시: ['00:00', '00:00', '00:00', '00:00', '00:00']
도착 시간 코드 | dtype: str | 예시: ['00:00', '00:00', '00:00', '00:00', '00:00']
이동 시간 | dtype: int64 | 예시: [4, 11, 9, 2, 2]

[거리] 관련 컬럼
이동거리 | dtype: int64 | 예시: [424, 1334, 250, 640, 320]

[이동량] 관련 컬럼

[인구] 관련 컬럼
이동 인구수 | dtype: float64 | 예시: [2.48, 4.04, 2.64, 1.21, 2.47]


In [27]:
# =========================================================
# 3. 출발·도착 시간 코드 확인
# =========================================================

START_TIME_COL = "출발 시간 코드"
END_TIME_COL = "도착 시간 코드"

for col in [START_TIME_COL, END_TIME_COL]:

    print("\n" + "=" * 60)
    print(col)
    print("=" * 60)

    print("dtype:", commute_raw[col].dtype)
    print("결측치:", f"{commute_raw[col].isna().sum():,}")
    print("고유값 개수:", commute_raw[col].nunique(dropna=True))

    time_counts = (
        commute_raw[col]
        .value_counts(dropna=False)
        .sort_index()
    )

    display(time_counts)


출발 시간 코드
dtype: str
결측치: 0
고유값 개수: 36


출발 시간 코드
00:00      103691
01:00      248477
02:00      221089
03:00      273024
04:00      690747
05:00     3632550
06:00    14858132
07:00    10623613
07:20    12633196
07:40    13613896
08:00    13288042
08:20    11116323
08:40     8765935
09:00     7333669
09:20     5561163
09:40     3968511
10:00     6748352
11:00     5331610
12:00     5217058
13:00     4327168
14:00     3258357
15:00     2504064
16:00     1662115
17:00      435917
17:20      364445
17:40      301700
18:00      256246
18:20      216038
18:40      194036
19:00      175304
19:20      158919
19:40      144076
20:00      315763
21:00      255667
22:00      176876
23:00       51317
Name: count, dtype: int64


도착 시간 코드
dtype: str
결측치: 0
고유값 개수: 36


도착 시간 코드
00:00       53223
01:00      157357
02:00      169510
03:00      183135
04:00      321379
05:00      776028
06:00     4872878
07:00     4402356
07:20     6453467
07:40     9582220
08:00    10823816
08:20    13852906
08:40    16906942
09:00    10276294
09:20     9290848
09:40     8853887
10:00    11522228
11:00     6268523
12:00     5512943
13:00     5254210
14:00     4020915
15:00     3086475
16:00     2423838
17:00      594506
17:20      498813
17:40      452166
18:00      331359
18:20      278184
18:40      251502
19:00      213548
19:20      187363
19:40      172697
20:00      361676
21:00      291765
22:00      237458
23:00       90671
Name: count, dtype: int64

In [28]:
# =========================================================
# 4. 시간 코드 형식 검사
# =========================================================

TIME_PATTERN = r"^(?:[01]\d|2[0-3]):(?:00|20|40)$"

for col in [START_TIME_COL, END_TIME_COL]:

    time_text = commute_raw[col].astype("string")

    invalid_mask = (
        time_text.notna()
        & ~time_text.str.match(TIME_PATTERN)
    )

    print(f"\n[{col}] 형식 오류")
    print("오류 행 수:", f"{invalid_mask.sum():,}")

    if invalid_mask.any():
        display(
            commute_raw.loc[
                invalid_mask,
                [col]
            ]
            .value_counts()
            .head(30)
        )


[출발 시간 코드] 형식 오류
오류 행 수: 0

[도착 시간 코드] 형식 오류
오류 행 수: 0


In [29]:
# 일반적인 HH:MM 형식 검사
GENERAL_TIME_PATTERN = r"^(?:[01]\d|2[0-3]):[0-5]\d$"

for col in [START_TIME_COL, END_TIME_COL]:

    time_text = commute_raw[col].astype("string")

    invalid_mask = (
        time_text.notna()
        & ~time_text.str.match(GENERAL_TIME_PATTERN)
    )

    print(
        col,
        "HH:MM 형식이 아닌 행:",
        f"{invalid_mask.sum():,}"
    )

    if invalid_mask.any():
        display(
            commute_raw.loc[
                invalid_mask,
                [col]
            ]
            .value_counts()
            .head(30)
        )

출발 시간 코드 HH:MM 형식이 아닌 행: 0
도착 시간 코드 HH:MM 형식이 아닌 행: 0


In [ ]:
# =========================================================
# 5. 주요 수치 변수 이상치 점검
# =========================================================

target_keywords = [
    "이동시간",
    "이동 시간",
    "소요시간",
    "이동거리",
    "이동 거리",
    "이동량"
]

target_cols = []

for col in commute_raw.columns:
    if any(keyword in col for keyword in target_keywords):
        target_cols.append(col)

print("점검 대상 컬럼:", target_cols)


outlier_summary = []

for col in target_cols:

    values = pd.to_numeric(
        commute_raw[col],
        errors="coerce"
    )

    valid = values.dropna()

    if valid.empty:
        continue

    q1 = valid.quantile(0.25)
    q3 = valid.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_summary.append({
        "컬럼": col,
        "유효값": valid.count(),
        "숫자 변환 실패": (
            commute_raw[col].notna().sum()
            - valid.count()
        ),
        "최솟값": valid.min(),
        "1%": valid.quantile(0.01),
        "중앙값": valid.median(),
        "99%": valid.quantile(0.99),
        "최댓값": valid.max(),
        "0 이하": (valid <= 0).sum(),
        "IQR 하한": lower,
        "IQR 상한": upper,
        "IQR 이상치 수": (
            (valid < lower) | (valid > upper)
        ).sum()
    })

outlier_summary = pd.DataFrame(outlier_summary)

display(outlier_summary)

점검 대상 컬럼: ['이동거리', '이동 시간']


,컬럼,유효값,숫자 변환 실패,최솟값,1%,중앙값,99%,최댓값,0 이하,IQR 하한,IQR 상한,IQR 이상치 수
0,이동거리,139027086,0,0,187.0,5661.0,22038.0,34384,1,-9172.5,21671.5,1599773
1,이동 시간,139027086,0,0,0.0,40.0,186.0,1334,1584431,-36.0,116.0,4702293


In [ ]:
# =========================================================
# 6. 행정동 코드 점검
# =========================================================

code_cols = [
    col for col in commute_raw.columns
    if "코드" in col and "시간" not in col
]

print("행정동·지역 코드 후보:", code_cols)

code_check = []

for col in code_cols:

    code_text = commute_raw[col].astype("string").str.strip()

    code_check.append({
        "컬럼": col,
        "결측치": code_text.isna().sum(),
        "빈 문자열": code_text.eq("").sum(),
        "고유값 수": code_text.nunique(dropna=True),
        "숫자 아닌 값": (
            code_text.notna()
            & ~code_text.str.fullmatch(r"\d+")
        ).sum(),
        "길이 종류": sorted(
            code_text.dropna()
            .str.len()
            .unique()
            .tolist()
        )
    })

display(pd.DataFrame(code_check))

출발동과 도착동이 같은 행: 6,038,950
내부 이동 비율: 4.3437%


In [34]:
# =========================================================
# 7. 출근 시간대 필터링
# 출발 또는 도착 시간이 07:00~09:40이면 남김
# =========================================================

START_TIME_COL = "출발 시간 코드"
END_TIME_COL = "도착 시간 코드"


def time_to_minutes(series):
    """
    HH:MM 문자열을 자정 이후 분 단위로 변환
    형식 오류는 NaN 처리
    """
    time_text = series.astype("string").str.strip()

    extracted = time_text.str.extract(
        r"^(?P<hour>\d{1,2}):(?P<minute>\d{2})$"
    )

    hour = pd.to_numeric(
        extracted["hour"],
        errors="coerce"
    )

    minute = pd.to_numeric(
        extracted["minute"],
        errors="coerce"
    )

    valid = (
        hour.between(0, 23)
        & minute.between(0, 59)
    )

    result = hour * 60 + minute

    return result.where(valid)


# 출발·도착 시간을 분으로 변환
start_minutes = time_to_minutes(
    commute_raw[START_TIME_COL]
)

end_minutes = time_to_minutes(
    commute_raw[END_TIME_COL]
)


# 오전 07:00~09:40
MORNING_START = 7 * 60
MORNING_END = 9 * 60 + 40


# 출발 시간이 범위에 포함
start_morning_mask = start_minutes.between(
    MORNING_START,
    MORNING_END,
    inclusive="both"
)

# 도착 시간이 범위에 포함
end_morning_mask = end_minutes.between(
    MORNING_START,
    MORNING_END,
    inclusive="both"
)


# 출발 또는 도착 중 하나라도 해당하면 유지
morning_mask = (
    start_morning_mask
    | end_morning_mask
)


commute_morning = (
    commute_raw.loc[morning_mask]
    .copy()
)


print("필터링 전:", f"{len(commute_raw):,}")
print("필터링 후:", f"{len(commute_morning):,}")
print(
    "유지 비율:",
    f"{len(commute_morning) / len(commute_raw) * 100:.2f}%"
)

필터링 전: 139,027,086
필터링 후: 100,172,925
유지 비율: 72.05%


In [35]:
# =========================================================
# 8. 필터링 조건 검증
# =========================================================

check_start_minutes = time_to_minutes(
    commute_morning[START_TIME_COL]
)

check_end_minutes = time_to_minutes(
    commute_morning[END_TIME_COL]
)

check_start_mask = check_start_minutes.between(
    MORNING_START,
    MORNING_END,
    inclusive="both"
)

check_end_mask = check_end_minutes.between(
    MORNING_START,
    MORNING_END,
    inclusive="both"
)


# 출발과 도착 모두 범위 밖인 행 확인
invalid_filtered_rows = commute_morning.loc[
    ~(check_start_mask | check_end_mask)
]

print(
    "필터 후 조건에 맞지 않는 행:",
    f"{len(invalid_filtered_rows):,}"
)

assert len(invalid_filtered_rows) == 0, (
    "출근 시간 범위 밖의 행이 남아 있습니다."
)

print("출근 시간대 필터 검증 완료")

필터 후 조건에 맞지 않는 행: 0
출근 시간대 필터 검증 완료


In [36]:
filter_type_summary = pd.Series({
    "출발만 시간대 포함": (
        start_morning_mask
        & ~end_morning_mask
    ).sum(),

    "도착만 시간대 포함": (
        ~start_morning_mask
        & end_morning_mask
    ).sum(),

    "출발·도착 모두 포함": (
        start_morning_mask
        & end_morning_mask
    ).sum(),

    "출발·도착 모두 제외": (
        ~start_morning_mask
        & ~end_morning_mask
    ).sum()
})

display(
    filter_type_summary
    .rename("행 수")
    .to_frame()
)

,행 수
출발만 시간대 포함,9730189
도착만 시간대 포함,13268577
출발·도착 모두 포함,77174159
출발·도착 모두 제외,38854161


In [37]:
print("필터 후 출발 시간 분포")
display(
    commute_morning[START_TIME_COL]
    .value_counts()
    .sort_index()
)

print("필터 후 도착 시간 분포")
display(
    commute_morning[END_TIME_COL]
    .value_counts()
    .sort_index()
)

필터 후 출발 시간 분포


출발 시간 코드
00:00       13877
01:00       54894
02:00       33319
03:00       34273
04:00      108517
05:00     1043994
06:00    11979703
07:00    10623613
07:20    12633196
07:40    13613896
08:00    13288042
08:20    11116323
08:40     8765935
09:00     7333669
09:20     5561163
09:40     3968511
Name: count, dtype: int64

필터 후 도착 시간 분포


도착 시간 코드
07:00     4402356
07:20     6453467
07:40     9582220
08:00    10823816
08:20    13852906
08:40    16906942
09:00    10276294
09:20     9290848
09:40     8853887
10:00     8334439
11:00      900595
12:00      244413
13:00      113580
14:00       61743
15:00       35212
16:00       20270
17:00        4133
17:20        3100
17:40        2450
18:00        2279
18:20        1580
18:40        1264
19:00         931
19:20         742
19:40         633
20:00        1322
21:00         837
22:00         588
23:00          78
Name: count, dtype: int64

In [38]:
from pathlib import Path

# 저장 경로
OUTPUT_DIR = PROCESSED_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "all_age_commute_morning_0700_0940.csv"

# 필터링 완료본 저장
commute_morning.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", OUTPUT_FILE)
print("파일 존재:", OUTPUT_FILE.exists())
print("저장 행 수:", f"{len(commute_morning):,}")
print(
    "파일 크기:",
    f"{OUTPUT_FILE.stat().st_size / 1024**3:.2f} GB"
)

저장 완료: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/all_age_commute_morning_0700_0940.csv
파일 존재: True
저장 행 수: 100,172,925
파일 크기: 11.40 GB
